# sqrt-eps-stabilize — ex2: contrast sqrt(var+eps) vs sqrt(var)+eps — gradient stability at var=0

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sqrt-eps-stabilize`. Running the final beacon cell reports progress against the `Numerical: sqrt-eps stabilization` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numerical: sqrt-eps stabilization` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sqrt-eps-stabilize`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sqrt-eps-stabilize"
DD_SUBTOPIC = "Numerical: sqrt-eps stabilization"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Where to place `eps` — INSIDE sqrt vs OUTSIDE

Ex1 used `sqrt(var + eps)` and showed it survives a zero-variance channel. The deepening move is the CONTRAST: what about `sqrt(var) + eps`? Same eps, just outside the sqrt.

```python
# INSIDE  — stable at var=0:
sigma_in  = (var + eps).sqrt()   # sqrt(eps) ≈ 3.16e-3 when eps=1e-5
# OUTSIDE — UNSTABLE at var=0:
sigma_out = var.sqrt() + eps     # sqrt(0) + eps = eps ≈ 1e-5
```

**Both are finite at var=0** — neither divides by zero — but they give VERY different normalizers. With `eps=1e-5` and a constant channel (var=0, x - mean = 0):
- inside:  `0 / 3.16e-3 = 0` — clean.
- outside: `0 / 1e-5 = 0` — also clean here, but...

**The OUTSIDE placement breaks the gradient.** `d/dvar sqrt(var)` is `1/(2*sqrt(var))` — infinite at `var=0`. Backprop through `sqrt(var) + eps` produces a non-finite gradient even though the forward value is finite. INSIDE placement keeps the derivative bounded because `d/dvar sqrt(var+eps) = 1/(2*sqrt(var+eps))` is at most `1/(2*sqrt(eps))`, never infinite.

**This is why PyTorch's BatchNorm/LayerNorm use INSIDE placement.** It's not about NaN in the forward — it's about NaN in the BACKWARD.

### Exercise 2 — contrast sqrt(var+eps) vs sqrt(var)+eps — gradient stability at var=0

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the two `eps`-placement variants — `sqrt(var + eps)` vs `sqrt(var) + eps` — by computing forward values and backward-pass gradients at `var=0`, and confirm the INSIDE placement bounds `d/dvar` while the OUTSIDE placement diverges.
> Keywords: sqrt, eps, gradient, batchnorm, stability
> ```

**KCs targeted:** `eps-inside-vs-outside-sqrt-forward`, `eps-inside-bounds-the-gradient`

Implement `ex2_compare_eps_placement(var, eps)`. The deepening variant of ex1.

Inputs:
- `var`: a 1-D `torch.Tensor` of variances (can include 0). Must be `float`, `requires_grad=False` going in (this function will build its own requires-grad variants for the gradient pass).
- `eps`: `float`.

Return a dict with EXACTLY these keys:

- `'sigma_inside'`: `torch.Tensor`, `(var + eps).sqrt()` — same shape as `var`, no_grad.
- `'sigma_outside'`: `torch.Tensor`, `var.sqrt() + eps`.
- `'sigma_inside_finite'`: `bool`, `torch.isfinite(sigma_inside).all().item()`.
- `'sigma_outside_finite'`: `bool`, `torch.isfinite(sigma_outside).all().item()`.
- `'grad_inside'`: `torch.Tensor`, the gradient `d sigma_inside / d var` at the given `var` values — computed by autograd on the inside expression with `.sum().backward()`.
- `'grad_outside'`: `torch.Tensor`, same for the outside expression. NOTE: at `var=0` this will be `inf` because the derivative of `sqrt(var)` at 0 is unbounded.
- `'grad_inside_finite'`: `bool`.
- `'grad_outside_finite'`: `bool`.
- `'inside_grad_upper_bound'`: `float`, `1.0 / (2 * sqrt(eps))` — the analytical max of `d sqrt(var + eps) / d var`, achieved at `var=0`.

Constraints:
- Use a FRESH `var.clone().detach().requires_grad_(True)` for each gradient computation so the two graphs don't share state.
- Always return shapes matching the input.

In [ ]:
def ex2_compare_eps_placement(var: Tensor, eps: float) -> dict:
    """Compare eps INSIDE vs OUTSIDE sqrt — forward + backward at var=0."""
    raise NotImplementedError()


def _test_ex2():
    import math

    # === Case 1: var contains a zero ===
    var = t.tensor([0.0, 0.25, 1.0, 4.0])
    eps = 1e-5
    d = ex2_compare_eps_placement(var, eps)

    # Forward values — both finite, but differ at var=0.
    assert d['sigma_inside_finite'] is True
    assert d['sigma_outside_finite'] is True
    assert math.isclose(d['sigma_inside'][0].item(), math.sqrt(eps), rel_tol=1e-6), (
        f'inside sqrt at var=0 should be sqrt(eps)={math.sqrt(eps)}, got {d["sigma_inside"][0].item()}'
    )
    assert math.isclose(d['sigma_outside'][0].item(), eps, rel_tol=1e-6), (
        f'outside sqrt at var=0 should be 0 + eps = {eps}, got {d["sigma_outside"][0].item()}'
    )
    # At var>0 the two are very close (just shifted by ~eps).
    for i in [1, 2, 3]:
        assert abs(d['sigma_inside'][i].item() - d['sigma_outside'][i].item()) < 1e-2, (
            f'at var>0 the two placements should be close; got '
            f'inside={d["sigma_inside"][i].item()}, outside={d["sigma_outside"][i].item()}'
        )

    # === Gradient at var=0: inside is BOUNDED, outside is INF ===
    assert d['grad_inside_finite'] is True, (
        f'gradient of sqrt(var+eps) must be finite at var=0; got grad_inside={d["grad_inside"]}'
    )
    assert d['grad_outside_finite'] is False, (
        f'gradient of sqrt(var)+eps must be NON-finite at var=0; got grad_outside={d["grad_outside"]}'
    )
    # Specifically, grad_outside[0] is inf or nan.
    assert not t.isfinite(d['grad_outside'][0]).item(), (
        f'd sqrt(var)/d var at var=0 must be inf; got {d["grad_outside"][0].item()}'
    )

    # === Analytical upper bound on inside gradient: 1/(2*sqrt(eps)) ===
    expected_bound = 1.0 / (2 * math.sqrt(eps))
    assert math.isclose(d['inside_grad_upper_bound'], expected_bound, rel_tol=1e-6)
    # Empirical grad_inside at var=0 should equal the bound (it's the max).
    assert math.isclose(d['grad_inside'][0].item(), expected_bound, rel_tol=1e-4), (
        f'grad_inside at var=0 should equal analytical bound {expected_bound}, got {d["grad_inside"][0].item()}'
    )
    # Empirical grad_inside at var>0 should be LESS THAN the bound.
    for i in [1, 2, 3]:
        g = d['grad_inside'][i].item()
        assert g < expected_bound, f'grad at var={var[i].item()} ({g}) should be < bound ({expected_bound})'

    # === Case 2: var all > 0 — both gradients finite ===
    var = t.tensor([0.1, 0.5, 1.0, 4.0])
    d = ex2_compare_eps_placement(var, eps=1e-5)
    assert d['grad_inside_finite'] is True
    assert d['grad_outside_finite'] is True
    # And the gradients are very close (the eps placement matters only near var=0).
    for i in range(4):
        gi = d['grad_inside'][i].item()
        go = d['grad_outside'][i].item()
        assert abs(gi - go) < 1e-3, f'far from var=0 the two grads should agree; got {gi} vs {go}'

    # === Case 3: shapes match input ===
    var = t.tensor([0.0, 0.0, 0.0])
    d = ex2_compare_eps_placement(var, eps=1e-4)
    assert d['sigma_inside'].shape == var.shape
    assert d['grad_inside'].shape == var.shape
    assert d['sigma_outside'].shape == var.shape
    assert d['grad_outside'].shape == var.shape
    # All three var=0 entries blow up on outside.
    assert d['grad_outside_finite'] is False

    # === All keys ===
    expected_keys = {'sigma_inside', 'sigma_outside',
                     'sigma_inside_finite', 'sigma_outside_finite',
                     'grad_inside', 'grad_outside',
                     'grad_inside_finite', 'grad_outside_finite',
                     'inside_grad_upper_bound'}
    assert set(d.keys()) == expected_keys, f'keys wrong: {set(d.keys())}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_compare_eps_placement(var, eps):
    import math
    # Forward (no grad needed here).
    with t.no_grad():
        sigma_inside = (var + eps).sqrt()
        sigma_outside = var.sqrt() + eps

    # Backward for inside.
    var_in = var.clone().detach().requires_grad_(True)
    ((var_in + eps).sqrt()).sum().backward()
    grad_inside = var_in.grad.detach().clone()

    # Backward for outside.
    var_out = var.clone().detach().requires_grad_(True)
    (var_out.sqrt() + eps).sum().backward()
    grad_outside = var_out.grad.detach().clone()

    return {
        'sigma_inside': sigma_inside,
        'sigma_outside': sigma_outside,
        'sigma_inside_finite': bool(t.isfinite(sigma_inside).all().item()),
        'sigma_outside_finite': bool(t.isfinite(sigma_outside).all().item()),
        'grad_inside': grad_inside,
        'grad_outside': grad_outside,
        'grad_inside_finite': bool(t.isfinite(grad_inside).all().item()),
        'grad_outside_finite': bool(t.isfinite(grad_outside).all().item()),
        'inside_grad_upper_bound': 1.0 / (2 * math.sqrt(eps)),
    }
```

**The forward is a red herring.** Both placements give finite forward values at `var=0` (`sqrt(eps)` vs `eps`). The stability argument is about the BACKWARD pass. `d/dvar sqrt(var) = 1/(2*sqrt(var))` blows up at 0; `d/dvar sqrt(var+eps) = 1/(2*sqrt(var+eps))` is bounded by `1/(2*sqrt(eps))`.

**Fresh `requires_grad_(True)` per pass.** Reusing the same tensor across two `.backward()` calls accumulates gradients into the SAME `.grad` field — you'd see the sum of both, not either one cleanly. `clone().detach().requires_grad_(True)` is the standard 'new graph leaf' pattern.

**Why BatchNorm/LayerNorm chose INSIDE.** Even in float32, early-training activations can produce variance very close to zero on collapsed channels. INSIDE placement means a max gradient of `1/(2*sqrt(1e-5)) ≈ 158` — large but FINITE. OUTSIDE placement would produce inf grads that propagate up the network and NaN-out the optimizer. The PyTorch convention is explicit in `aten/src/ATen/native/Normalization.cpp`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()